# 3_Mesh to Image 
Here we take the cleaned mesh output from Meshmixer, and we convert it to an isotropic volume with voxel resolution that we specify. Then we save the image. 

In [1]:
from vedo import Mesh, Plotter, Volume
import vtk 
from skimage.morphology import skeletonize 
from skimage import data 
from skimage.morphology import medial_axis, skeletonize, skeletonize_3d
from scipy.ndimage import zoom
from scipy import ndimage
import tifffile as tf
from skimage.util import map_array 
import os
import pymeshlab
import cv2
import concurrent.futures
import time
import numpy as np
from tqdm import tqdm

Change the necessary parameters in the next cell. 

In [2]:
voxel = input("Target voxel size:") #this is in the same unit as your mesh 
# With script from https://forum.image.sc/t/create-2d-tiff-stack-from-3d-model/67419 
# Specify the input folder containing OBJ files
folder_path = r"F:\Megumi\Dropbox (DBOX-EQS1)\Megumi-Cadisha\Skeleton by Lobe\Left Upper Lobe\proximal_pruning\EH3419LU-P9.1(9.1)\\"
sample_ID =  input("ID: ") 
input_obj = folder_path + f"{sample_ID}.obj" 
output_tif = folder_path + f"{sample_ID}_{voxel}um.tif" 
measurements_txt = folder_path + f"{sample_ID}_measurements.txt"

Target voxel size: 7
ID:  EH3419LU


The actual processing steps are in the next cell. The input obj will be converted to a binary image with specified voxel sizes in um units. A window will pop up to show the input surface (obj) on the left with the binarized volume on the right. You can turn around the object and inspect it to make sure that it is free of artifacts at this stage (while rare, there may be missing pieces of the object, or spurious artifacts from imperfect surfaces). If you see such artifacts, you may want to check the input surface file and fix it. This will also measure the (1) surface area of the mesh, (2) mesh volume, and (3) binary image volume amd save ot omtp a txt file called measurements.txt. 

In [3]:
def timed_future_progress_bar(future, expected_time, increments=10):
    """
    Display progress bar for expected_time seconds.
    
    """
    interval = expected_time / increments
    with tqdm(total=increments) as pbar:
        for i in range(increments - 1):
            if future.done():
                # finish the progress bar
                # not sure if there's a cleaner way to do this?
                pbar.update(increments - i)
                return
            else:
                time.sleep(interval)
                pbar.update()
        # if the future still hasn't completed, wait for it.
        future.result()
        pbar.update()


def blocking_job():
    # We take the input obj file and extract the surface area and volume 
    input_obj = folder_path + f"{sample_ID}.obj" 
    print('Loading...')
    surf = Mesh(input_obj)
    wireframe = surf.wireframe()
    area = surf.area()
    mesh_vol = surf.volume()
    
    # Next, binarize the surface at specified voxel resolution 
    int_voxel = int(voxel)
    vol = surf.binarize(spacing=(int_voxel,int_voxel,int_voxel))
    
    
    # The isovolume is the new volume with isotropic voxels 
    # This value is larger than the input mesh volume 
    iso = vol.isosurface().color("blue5")

    
    #vol.write(output_tif) 
    array = vol.tonumpy() 
    array = array[..., np.newaxis]
    array = array.transpose(2,3,1,0)
    tf.imwrite(output_tif, array, imagej=True,
               resolution=(1./int_voxel, 1./int_voxel),
               metadata={
                   'spacing': int_voxel,
                   'unit': 'um'
               }
              )    
    
    ### To visualize the isosurface ###
    # This is useful to make sure that our file is not corrupt, and that we are not missing any branches. 
    plt = Plotter(N=2, axes=9)
    plt.at(0).show(wireframe, __doc__)
    plt.at(1).show("..the volume is isosurfaced:", iso)
    plt.interactive().close()
    
    # Save measurements into the text file 
    with open(measurements_txt, "w") as file: 
        file.write("\n" + "Mesh area is:" + str(area))
        file.write("\n" + "Mesh volume is:" + str(mesh_vol))
        file.write("\n" + "Iso volume is:" + str(iso.volume()))
    
    return f'Mesh area is:{area} \nMesh volume is:{mesh_vol} \nIsovolume is:{iso.volume()} \nDone.'


with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    future = pool.submit(blocking_job)
    timed_future_progress_bar(future, 5)
print(f'Work done: {future.result()}')

Loading...


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [03:02<00:00, 18.24s/it]

Work done: Mesh area is:60979993.24171005 
Mesh volume is:1252774710.5104089 
Isovolume is:1302390612.353587 
Done.
